In [32]:

import pandas as pd
from rdkit import Chem
import torch
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
from sklearn.metrics import roc_auc_score
from torch_geometric.loader import DataLoader
import numpy as np
import random
from itertools import product
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score,
    balanced_accuracy_score,
    f1_score,
    recall_score,
    average_precision_score,
    matthews_corrcoef
)
from sklearn.model_selection import train_test_split
import sys
import os
sys.path.append(os.path.abspath('../'))
from reduceGraph import mol_to_graph, graph_to_pyg_oh, reduce_graph_from_mol_oh, mol_to_pool_idx
from networks import GAT, PPGAT

In [33]:
#set random seeds
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)


In [3]:
#DATASET 

dataset = pd.read_csv('clintox.csv')
dataset

,smiles,FDA_APPROVED,CT_TOX
0,*C(=O)[C@H](CCCCNC(=O)OCCOC)NC(=O)OCCOC,1,0
1,[C@@H]1([C@@H]([C@@H]([C@H]([C@@H]([C@@H]1Cl)C...,1,0
2,[C@H]([C@@H]([C@@H](C(=O)[O-])O)O)([C@H](C(=O)...,1,0
3,[H]/[NH+]=C(/C1=CC(=O)/C(=C\C=c2ccc(=C([NH3+])...,1,0
4,[H]/[NH+]=C(\N)/c1ccc(cc1)OCCCCCOc2ccc(cc2)/C(...,1,0
...,...,...,...
1479,O[Si](=O)O,1,0
1480,O=[Ti]=O,1,0
1481,O=[Zn],1,0
1482,OCl(=O)(=O)=O,1,0


In [4]:
dataset.FDA_APPROVED.value_counts()

FDA_APPROVED
1    1390
0      94
Name: count, dtype: int64

In [5]:
dataset.CT_TOX.value_counts()

CT_TOX
0    1372
1     112
Name: count, dtype: int64

In [6]:
tox = dataset['CT_TOX']
tox.value_counts()

CT_TOX
0    1372
1     112
Name: count, dtype: int64

In [7]:
def smiles_to_data(smiles, tox_label):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None  
    graph = mol_to_graph(mol)   # mol to networkx 
    data = graph_to_pyg_oh(graph)  # networx graph to pytorch geometric graph
    tox_label_tensor = torch.tensor([tox_label], dtype=torch.float) 
    data.y = torch.stack([tox_label_tensor], dim=1)

    return data


def dataframe_to_pyg_dataset(df, smiles_col, tox_label_col):
    data_list = []
    
    for idx, row in df.iterrows():
        smiles = row[smiles_col]
        tox_label = row[tox_label_col]
        data = smiles_to_data(smiles, tox_label)
        if data is not None:
            data.smiles = smiles
            data_list.append(data)

    return data_list


In [8]:
clintox_dataset = dataframe_to_pyg_dataset(dataset, 'smiles', 'CT_TOX')


[11:36:41] Explicit valence for atom # 0 N, 5, is greater than permitted
[11:36:42] Can't kekulize mol.  Unkekulized atoms: 9
[11:36:43] Explicit valence for atom # 10 N, 4, is greater than permitted
[11:36:43] Explicit valence for atom # 10 N, 4, is greater than permitted
[11:36:43] Can't kekulize mol.  Unkekulized atoms: 4
[11:36:43] Can't kekulize mol.  Unkekulized atoms: 4


In [9]:
#grid search
learning_rates = [1e-4, 5e-4, 1e-3]
batch_sizes = [16, 32, 64]
grid = list(product(learning_rates, batch_sizes))

In [11]:


def train_eval_model(mod, dataset, in_channels, edge_attr_dim, out_channels, grid, epochs=30):
    labels = [int(data.y.item()) for data in dataset]

    train_val_indices, test_indices = train_test_split(
        list(range(len(dataset))),
        test_size=0.2,
        stratify=labels,
        random_state=42
    )

    train_val_labels = [labels[i] for i in train_val_indices]

    # 10% of full = 12.5% of 80% → 0.125
    train_indices, val_indices = train_test_split(
        train_val_indices,
        test_size=0.125,
        stratify=train_val_labels,
        random_state=42
    )

    # Create subsets 
    train = [dataset[i] for i in train_indices]
    val = [dataset[i] for i in val_indices]
    test  = [dataset[i] for i in test_indices]


    #  Hyperparameter tuning 
    best_val_loss = float('inf')
    best_config = None
    best_model_state = None

    for lr, batch_size in grid:
        model = mod(in_channels, edge_attr_dim, hidden_channels=64, out_channels=out_channels).to('cuda')
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        criterion = torch.nn.BCEWithLogitsLoss()

        train_loader = DataLoader(train, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val, batch_size=batch_size)

        for epoch in range(epochs):
            model.train()
            for batch in train_loader:
                batch = batch.to('cuda')
                optimizer.zero_grad()
                out = model(batch)
                loss = criterion(out, batch.y.float())
                loss.backward()
                optimizer.step()

        # Evaluate on val
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch in val_loader:
                batch = batch.to('cuda')
                out = model(batch)
                val_loss += criterion(out, batch.y.float()).item() * batch.num_graphs


        val_loss /= len(val)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_config = (lr, batch_size)
            best_model_state = model.state_dict()

    #  Retrain final model on train + val 
    final_model = mod(in_channels, edge_attr_dim, hidden_channels=64, out_channels=out_channels).to('cuda')
    final_model.load_state_dict(best_model_state)

    full_train = train + val
    train_loader = DataLoader(full_train, batch_size=best_config[1], shuffle=True)
    optimizer = torch.optim.Adam(final_model.parameters(), lr=best_config[0])
    criterion = torch.nn.BCEWithLogitsLoss()

    for epoch in range(epochs):
        final_model.train()
        for batch in train_loader:
            batch = batch.to('cuda')
            optimizer.zero_grad()
            out = final_model(batch)
            loss = criterion(out, batch.y.float())
            loss.backward()
            optimizer.step()

    return final_model, best_config, best_val_loss

In [12]:
in_channels = clintox_dataset[0].x.size(-1)
edge_attr_dim = clintox_dataset[0].edge_attr.size(-1)
out_channels = 1

model, config, loss = train_eval_model(GAT, clintox_dataset, in_channels, edge_attr_dim, out_channels, grid)

print(config )





(0.001, 64)


In [14]:
def cross_validate_stratified(
    mod,
    dataset, in_channels, edge_attr_dim, out_channels,
    best_lr, best_batch_size, epochs=30, k=5
):
    from sklearn.model_selection import StratifiedKFold
    from sklearn.metrics import roc_auc_score, balanced_accuracy_score
    import numpy as np
    import torch

    # Extract single target per sample
    y_vector = torch.stack([data.y.view(-1)[0] for data in dataset]).numpy()

    splitter = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)

    all_aurocs = []
    all_bal_accs = []
    all_f1s = []
    all_sensitivities = []
    all_specificities = []
    all_pr_aucs = []
    all_mccs = []

    for fold, (train_val_idx, test_idx) in enumerate(splitter.split(np.zeros(len(y_vector)), y_vector)):
        print(f"\n Fold {fold + 1}")

        # Split data
        y_train_val = y_vector[train_val_idx]
        train_val_data = [dataset[i] for i in train_val_idx]
        test_data = [dataset[i] for i in test_idx]

        # Inner val split
        inner_split = StratifiedKFold(n_splits=5, shuffle=True, random_state=fold)
        train_idx, val_idx = next(inner_split.split(np.zeros(len(y_train_val)), y_train_val))
        train = [train_val_data[i] for i in train_idx]
        val = [train_val_data[i] for i in val_idx]

        # Initialize model
        model = mod(in_channels, edge_attr_dim, hidden_channels=64, out_channels=out_channels).to('cuda')
        optimizer = torch.optim.Adam(model.parameters(), lr=best_lr)
        criterion = torch.nn.BCEWithLogitsLoss()

        train_loader = DataLoader(train, batch_size=best_batch_size, shuffle=True)
        val_loader = DataLoader(val, batch_size=best_batch_size)
        test_loader = DataLoader(test_data, batch_size=best_batch_size)

        # Training loop
        for epoch in range(epochs):
            model.train()
            for batch in train_loader:
                batch = batch.to('cuda')
                optimizer.zero_grad()
                out = model(batch)

                y_true = batch.y.float()
                if y_true.dim() == 1:
                    y_true = y_true.unsqueeze(1)

                loss = criterion(out, y_true)
                loss.backward()
                optimizer.step()

        # Evaluate on test set
        model.eval()
        y_true, y_scores = [], []

        with torch.no_grad():
            for batch in test_loader:
                batch = batch.to('cuda')
                out = model(batch)

                y_scores.append(out.cpu())
                y_true.append(batch.y.cpu())

        y_scores = torch.cat(y_scores).numpy()
        y_true = torch.cat(y_true).numpy()

        # AUROC
        auroc = roc_auc_score(y_true, y_scores)
        all_aurocs.append(auroc)

        # Convert logits → probabilities → predictions
        y_probs = 1 / (1 + np.exp(-y_scores))   # sigmoid
        y_pred = (y_probs >= 0.5).astype(int)

        # Balanced accuracy
        bal_acc = balanced_accuracy_score(y_true, y_pred)
        all_bal_accs.append(bal_acc)

        # F1 score
        f1 = f1_score(y_true, y_pred)
        all_f1s.append(f1)

        # Sensitivity (Recall / True Positive Rate)
        sensitivity = recall_score(y_true, y_pred)
        all_sensitivities.append(sensitivity)

        # Specificity (True Negative Rate)
        specificity = recall_score(y_true, y_pred, pos_label=0)
        all_specificities.append(specificity)

        # PR-AUC (Average Precision)
        pr_auc = average_precision_score(y_true, y_scores)
        all_pr_aucs.append(pr_auc)

        # Matthews Correlation Coefficient (MCC)
        mcc = matthews_corrcoef(y_true, y_pred)
        all_mccs.append(mcc)

        print(f" Fold {fold + 1} AUROC: {auroc:.4f}")
        print(f" Fold {fold + 1} Balanced Acc: {bal_acc:.4f}")

    mean_auroc = np.mean(all_aurocs)
    std_auroc = np.std(all_aurocs)

    mean_bal_acc = np.mean(all_bal_accs)
    std_bal_acc = np.std(all_bal_accs)

    mean_f1 = np.mean(all_f1s)
    std_f1 = np.std(all_f1s)

    mean_sensitivity = np.mean(all_sensitivities)
    std_sensitivity = np.std(all_sensitivities)

    mean_specificity = np.mean(all_specificities)
    std_specificity = np.std(all_specificities)

    mean_pr_auc = np.mean(all_pr_aucs)
    std_pr_auc = np.std(all_pr_aucs)

    mean_mcc = np.mean(all_mccs)
    std_mcc = np.std(all_mccs)

    print(f"\n Mean AUROC = {mean_auroc:.4f} ± {std_auroc:.4f}")
    print(f" Mean Balanced Accuracy = {mean_bal_acc:.4f} ± {std_bal_acc:.4f}")
    print(f"\n Mean F1 = {mean_f1:.4f} ± {std_f1:.4f}")
    print(f"\n Mean sensitivity = {mean_sensitivity:.4f} ± {std_sensitivity:.4f}")
    print(f"\n Mean specificity = {mean_specificity:.4f} ± {std_specificity:.4f}")
    print(f"\n Mean PR-AUC = {mean_pr_auc:.4f} ± {std_pr_auc:.4f}")
    print(f"\n Mean MCC= {mean_mcc:.4f} ± {std_mcc:.4f}")

    return (
        all_aurocs, mean_auroc, std_auroc,
        all_bal_accs, mean_bal_acc, std_bal_acc
    )

In [15]:
all_aurocs, mean_aurocs, std_aurocs, all_bal_accs, mean_bal_accs, std_bal_accs = cross_validate_stratified(
    GAT,
    clintox_dataset,
    in_channels,
    edge_attr_dim,
    out_channels=1,
    best_lr=config[0],
    best_batch_size=config[1],
    epochs=100,
    k=5
)


 Fold 1
 Fold 1 AUROC: 0.8635
 Fold 1 Balanced Acc: 0.6735

 Fold 2
 Fold 2 AUROC: 0.8839
 Fold 2 Balanced Acc: 0.7064

 Fold 3


/tmp/ipykernel_169230/1645237278.py:83: RuntimeWarning: overflow encountered in exp
  y_probs = 1 / (1 + np.exp(-y_scores))   # sigmoid


 Fold 3 AUROC: 0.8372
 Fold 3 Balanced Acc: 0.6174

 Fold 4


/tmp/ipykernel_169230/1645237278.py:83: RuntimeWarning: overflow encountered in exp
  y_probs = 1 / (1 + np.exp(-y_scores))   # sigmoid


 Fold 4 AUROC: 0.8573
 Fold 4 Balanced Acc: 0.6752

 Fold 5
 Fold 5 AUROC: 0.8575
 Fold 5 Balanced Acc: 0.6881

 Mean AUROC = 0.8599 ± 0.0149
 Mean Balanced Accuracy = 0.6721 ± 0.0298

 Mean F1 = 0.4013 ± 0.0764

 Mean sensitivity = 0.3933 ± 0.0456

 Mean specificity = 0.9510 ± 0.0183

 Mean PR-AUC = 0.4640 ± 0.0938

 Mean MCC= 0.3551 ± 0.0901


# RG

In [16]:
def smiles_to_rgdata(smiles, tox_label):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None  
    
    data = reduce_graph_from_mol_oh(mol)  # Convert to PyG reduced graph
    
    # Skip if edge_attr is missing 
    if data.edge_attr is None or data.edge_attr.dim() != 2 or data.edge_attr.size(0) == 0 or data.edge_attr.size(1) == 0:
        return None

    tox_label_tensor = torch.tensor([tox_label], dtype=torch.float) 
    data.y = torch.stack([tox_label_tensor], dim=1)
    return data

def dataframe_to_rg_pyg_dataset(df, smiles_col, tox_label_col):
    data_list = []

    for idx, row in df.iterrows():
        smiles = row[smiles_col]
        tox_label = row[tox_label_col]
        data = smiles_to_rgdata(smiles, tox_label)
        if data is not None:
            data.smiles = smiles
            data_list.append(data)

    return data_list

In [17]:
clintox_rg_dataset = dataframe_to_rg_pyg_dataset(dataset, 'smiles', 'CT_TOX')


[11:45:56] Explicit valence for atom # 0 N, 5, is greater than permitted
[11:45:57] Can't kekulize mol.  Unkekulized atoms: 9
[11:46:01] Explicit valence for atom # 10 N, 4, is greater than permitted
[11:46:01] Explicit valence for atom # 10 N, 4, is greater than permitted
[11:46:03] Can't kekulize mol.  Unkekulized atoms: 4
[11:46:03] Can't kekulize mol.  Unkekulized atoms: 4


In [18]:

in_channels = clintox_dataset[0].x.size(-1)
edge_attr_dim = clintox_dataset[0].edge_attr.size(-1)
out_channels = 1

model, config, loss = train_eval_model(GAT, clintox_rg_dataset, in_channels, edge_attr_dim, out_channels, grid)

print(config )


(0.001, 32)


In [19]:
all_aurocs, mean_aurocs, std_aurocs, all_bal_accs, mean_bal_accs, std_bal_accs = cross_validate_stratified(
    GAT,
    clintox_rg_dataset,
    in_channels,
    edge_attr_dim,
    out_channels=1,
    best_lr=config[0],
    best_batch_size=config[1],
    epochs=100,
    k=5
)


 Fold 1


/tmp/ipykernel_169230/1645237278.py:83: RuntimeWarning: overflow encountered in exp
  y_probs = 1 / (1 + np.exp(-y_scores))   # sigmoid


 Fold 1 AUROC: 0.7289
 Fold 1 Balanced Acc: 0.5707

 Fold 2
 Fold 2 AUROC: 0.7942
 Fold 2 Balanced Acc: 0.6242

 Fold 3
 Fold 3 AUROC: 0.7618
 Fold 3 Balanced Acc: 0.5572

 Fold 4


/tmp/ipykernel_169230/1645237278.py:83: RuntimeWarning: overflow encountered in exp
  y_probs = 1 / (1 + np.exp(-y_scores))   # sigmoid


 Fold 4 AUROC: 0.8022
 Fold 4 Balanced Acc: 0.6643

 Fold 5
 Fold 5 AUROC: 0.7317
 Fold 5 Balanced Acc: 0.5516

 Mean AUROC = 0.7638 ± 0.0305
 Mean Balanced Accuracy = 0.5936 ± 0.0437

 Mean F1 = 0.2490 ± 0.0710

 Mean sensitivity = 0.2328 ± 0.1035

 Mean specificity = 0.9543 ± 0.0180

 Mean PR-AUC = 0.2486 ± 0.0494

 Mean MCC= 0.2043 ± 0.0610


/tmp/ipykernel_169230/1645237278.py:83: RuntimeWarning: overflow encountered in exp
  y_probs = 1 / (1 + np.exp(-y_scores))   # sigmoid


# PPGAT

In [20]:
def smiles_to_rgnn_data(smiles, tox_label):
    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return None  
    G = mol_to_graph(mol)   # mol to networkx 
    data = graph_to_pyg_oh(G)  # networx graph to pytorch geometric graph
    tox_label_tensor = torch.tensor([tox_label], dtype=torch.float) 
    data.y = torch.stack([tox_label_tensor], dim=1)
    
    pharma_index, new_edge_index, new_edge_attr = mol_to_pool_idx(mol)
    data.pharma_index = pharma_index
    data.new_edge_index = new_edge_index
    data.new_edge_attr = new_edge_attr

    # Skip if edge_attr is missing 
    if data.edge_attr is None or data.edge_attr.dim() != 2 or data.edge_attr.size(0) == 0 or data.edge_attr.size(1) == 0:
        return None
    
    if data.edge_index is None or data.edge_index.dim() != 2 or data.edge_index.size(0) == 0 or data.edge_index.size(1) == 0:
        return None

 

    return data

def dataframe_to_rgnn_pyg_dataset(df, smiles_col, tox_label_col):
    data_list = []

    for idx, row in df.iterrows():
        smiles = row[smiles_col]
    
        tox_label = row[tox_label_col]
        data = smiles_to_rgnn_data(smiles, tox_label)
        if data is not None:
            data.smiles = smiles
            data_list.append(data)

    return data_list

In [21]:
clintox_ppgat_dataset = dataframe_to_rgnn_pyg_dataset(dataset, 'smiles',  'CT_TOX')


[11:52:21] Explicit valence for atom # 0 N, 5, is greater than permitted
[11:52:22] Can't kekulize mol.  Unkekulized atoms: 9
[11:52:26] Explicit valence for atom # 10 N, 4, is greater than permitted
[11:52:26] Explicit valence for atom # 10 N, 4, is greater than permitted
[11:52:27] Can't kekulize mol.  Unkekulized atoms: 4
[11:52:27] Can't kekulize mol.  Unkekulized atoms: 4


In [22]:

in_channels = clintox_ppgat_dataset[0].x.size(-1)
edge_attr_dim = clintox_ppgat_dataset[0].edge_attr.size(-1)
out_channels = 1

model, config, loss = train_eval_model(PPGAT, clintox_ppgat_dataset, in_channels, edge_attr_dim, out_channels, grid)

print(config )


(0.0005, 32)


In [23]:
all_aurocs, mean_aurocs, std_aurocs, all_bal_accs, mean_bal_accs, std_bal_accs = cross_validate_stratified(
    PPGAT,
    clintox_ppgat_dataset,
    in_channels,
    edge_attr_dim,
    out_channels=1,
    best_lr=config[0],
    best_batch_size=config[1],
    epochs=100,
    k=5
)


 Fold 1
 Fold 1 AUROC: 0.8871
 Fold 1 Balanced Acc: 0.6309

 Fold 2
 Fold 2 AUROC: 0.8242
 Fold 2 Balanced Acc: 0.6629

 Fold 3
 Fold 3 AUROC: 0.8138
 Fold 3 Balanced Acc: 0.6771

 Fold 4


/tmp/ipykernel_169230/1645237278.py:83: RuntimeWarning: overflow encountered in exp
  y_probs = 1 / (1 + np.exp(-y_scores))   # sigmoid


 Fold 4 AUROC: 0.8734
 Fold 4 Balanced Acc: 0.7145

 Fold 5
 Fold 5 AUROC: 0.8251
 Fold 5 Balanced Acc: 0.7071

 Mean AUROC = 0.8447 ± 0.0296
 Mean Balanced Accuracy = 0.6785 ± 0.0304

 Mean F1 = 0.4378 ± 0.0472

 Mean sensitivity = 0.3877 ± 0.0696

 Mean specificity = 0.9692 ± 0.0153

 Mean PR-AUC = 0.4865 ± 0.0625

 Mean MCC= 0.4116 ± 0.0459


In [24]:
def evaluate_model(model, dataloader, device):
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for data in dataloader:
            data = data.to(device)
            out = model(data)
            probs = torch.sigmoid(out).view(-1).cpu().numpy()
            labels = data.y.view(-1).cpu().numpy()

            # Ensure probs and labels are arrays, even for batch size 1
            probs = np.atleast_1d(probs)
            labels = np.atleast_1d(labels)

            all_probs.extend(probs)
            all_labels.extend(labels)

    y_pred = (np.array(all_probs) > 0.5).astype(int)
    acc = balanced_accuracy_score(all_labels, y_pred)
    auroc = roc_auc_score(all_labels, all_probs)
    return acc, auroc

In [34]:
#For random splits 
#for statistical test: STORE ALL RUN RESULTS 
GAT_acc_runs_random = {}
GAT_rg_acc_runs_random = {}
PPGAT_acc_runs_random = {}

GAT_auroc_runs_random = {}
GAT_rg_auroc_runs_random = {}
PPGAT_auroc_runs_random = {}


trials = 5


acc_scores = []
auroc_scores = []

acc_scores_rg = []
auroc_scores_rg= []

acc_scores_ppgat = []
auroc_scores_ppgat = []

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")



for run in range(trials):
    print("RUN:", run)
        
    n = len(clintox_rg_dataset)
    indices = np.arange(n)

    #independent run
    np.random.seed(42 + run) 
    np.random.shuffle(indices)

    split_80 = int(0.8 * n)
    train_val_idx = indices[:split_80]
    test_idx = indices[split_80:]
    split_90 = int(0.9 * len(train_val_idx))
    train_idx = train_val_idx[:split_90]
    val_idx = train_val_idx[split_90:]


    # GAT 
    train_set = [clintox_dataset[i] for i in train_idx]
    val_set   = [clintox_dataset[i] for i in val_idx]
    test_set  = [clintox_dataset[i] for i in test_idx]

    # RG 
    train_rg = [clintox_rg_dataset[i] for i in train_idx]
    val_rg   = [clintox_rg_dataset[i] for i in val_idx]
    test_rg  = [clintox_rg_dataset[i] for i in test_idx]

    # PPGAT 
    train_ppgat = [clintox_ppgat_dataset[i] for i in train_idx]
    val_ppgat   = [clintox_ppgat_dataset[i] for i in val_idx]
    test_ppgat  = [clintox_ppgat_dataset[i] for i in test_idx]

    # GAT
    print("GAT")


    lr = config[0]
    batch_size = config[1]

    train_loader = DataLoader(train_set, batch_size=batch_size)
    test_loader  = DataLoader(test_set, batch_size=batch_size)
    val_loader   = DataLoader(val_set, batch_size=batch_size)

    model = GAT(
        in_channels=train_set[0].x.shape[1],
        edge_attr_dim=train_set[0].edge_attr.shape[1] if train_set[0].edge_attr is not None else 0,
        hidden_channels=64,
        out_channels=1,
        heads=4
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = torch.nn.BCEWithLogitsLoss()

    for epoch in range(1, 101):
        model.train()
        for data in train_loader:
            data = data.to(device)
            optimizer.zero_grad()
            loss = criterion(model(data), data.y.float())
            loss.backward()
            optimizer.step()

    final_acc, final_auroc = evaluate_model(model, test_loader, device)
    acc_scores.append(final_acc)
    auroc_scores.append(final_auroc)

    # GAT-rg
    print("GAT-rg")

    
    lr = config[0]
    batch_size = config[1]

    train_loader = DataLoader(train_rg, batch_size=batch_size)
    test_loader  = DataLoader(test_rg, batch_size=batch_size)

    model = GAT(
        in_channels=train_rg[0].x.shape[1],
        edge_attr_dim=train_rg[0].edge_attr.shape[1] if train_rg[0].edge_attr is not None else 0,
        hidden_channels=64,
        out_channels=1,
        heads=4
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(1, 101):
        model.train()
        for data in train_loader:
            data = data.to(device)
            optimizer.zero_grad()
            loss = criterion(model(data), data.y.float())
            loss.backward()
            optimizer.step()

    final_acc, final_auroc = evaluate_model(model, test_loader, device)
    acc_scores_rg.append(final_acc)
    auroc_scores_rg.append(final_auroc)

    # PPGAT
    print("PPGAT")

    lr = config[0]
    batch_size = config[1]

    train_loader = DataLoader(train_ppgat, batch_size=batch_size)
    test_loader  = DataLoader(test_ppgat, batch_size=batch_size)

    model = PPGAT(
        in_channels=train_ppgat[0].x.shape[1],
        edge_attr_dim=train_ppgat[0].edge_attr.shape[1] if train_ppgat[0].edge_attr is not None else 0,
        hidden_channels=64,
        out_channels=1,
        heads=4
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(1, 101):
        model.train()
        for data in train_loader:
            data = data.to(device)
            optimizer.zero_grad()
            loss = criterion(model(data), data.y.float())
            loss.backward()
            optimizer.step()

    final_acc, final_auroc = evaluate_model(model, test_loader, device)
    acc_scores_ppgat.append(final_acc)
    auroc_scores_ppgat.append(final_auroc)


# Store runs (for stat test)
GAT_acc_runs_random["clintox"] = acc_scores
GAT_auroc_runs_random["clintox"] = auroc_scores

GAT_rg_acc_runs_random["clintox"] = acc_scores_rg
GAT_rg_auroc_runs_random["clintox"] = auroc_scores_rg

PPGAT_acc_runs_random["clintox"] = acc_scores_ppgat
PPGAT_auroc_runs_random["clintox"] = auroc_scores_ppgat

#SAVE
rows = []

for run in range(trials):
    rows.append({
        "run": run,

        "GAT_acc": GAT_acc_runs_random["clintox"][run],
        "GAT_auroc": GAT_auroc_runs_random["clintox"][run],

        "GAT_rg_acc": GAT_rg_acc_runs_random["clintox"][run],
        "GAT_rg_auroc": GAT_rg_auroc_runs_random["clintox"][run],

        "PPGAT_acc": PPGAT_acc_runs_random["clintox"][run],
        "PPGAT_auroc": PPGAT_auroc_runs_random["clintox"][run],
    })

runs_df = pd.DataFrame(rows)

#runs_df.to_csv("random_spli_BBBP_runs.csv", index=False)

# FINAL TABLE
results = []

results.append({

    "GAT_acc_mean": np.mean(GAT_acc_runs_random["clintox"]),
    "GAT_acc_std":  np.std(GAT_acc_runs_random["clintox"]),
    "GAT_auroc_mean": np.mean(GAT_auroc_runs_random["clintox"]),
    "GAT_auroc_std":  np.std(GAT_auroc_runs_random["clintox"]),

    "GAT_rg_acc_mean": np.mean(GAT_rg_acc_runs_random["clintox"]),
    "GAT_rg_acc_std":  np.std(GAT_rg_acc_runs_random["clintox"]),
    "GAT_rg_auroc_mean": np.mean(GAT_rg_auroc_runs_random["clintox"]),
    "GAT_rg_auroc_std":  np.std(GAT_rg_auroc_runs_random["clintox"]),

    "PPGAT_acc_mean": np.mean(PPGAT_acc_runs_random["clintox"]),
    "PPGAT_acc_std":  np.std(PPGAT_acc_runs_random["clintox"]),
    "PPGAT_auroc_mean": np.mean(PPGAT_auroc_runs_random["clintox"]),
    "PPGAT_auroc_std":  np.std(PPGAT_auroc_runs_random["clintox"]),
})

results_df = pd.DataFrame(results)
print(results_df)


RUN: 0
GAT
GAT-rg
PPGAT
RUN: 1
GAT
GAT-rg
PPGAT
RUN: 2
GAT
GAT-rg
PPGAT
RUN: 3
GAT
GAT-rg
PPGAT
RUN: 4
GAT
GAT-rg
PPGAT
   GAT_acc_mean  GAT_acc_std  GAT_auroc_mean  GAT_auroc_std  GAT_rg_acc_mean  \
0      0.689676     0.052539         0.89304       0.028434         0.581305   

   GAT_rg_acc_std  GAT_rg_auroc_mean  GAT_rg_auroc_std  PPGAT_acc_mean  \
0        0.038572           0.769125          0.032166         0.66705   

   PPGAT_acc_std  PPGAT_auroc_mean  PPGAT_auroc_std  
0        0.03593          0.864345         0.027324  
